In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive"
PROJECT = f"{DRIVE}/abaya_bot_project"
import os
os.makedirs(PROJECT, exist_ok=True)

Mounted at /content/drive


In [ ]:
!pip install -q sentence-transformers pillow numpy

In [ ]:
import json
import numpy as np
from PIL import Image

catalog = json.load(open(f"{PROJECT}/catalog.json", encoding="utf-8"))
imgs = [Image.open(f"{PROJECT}/images/{c['image']}").convert("RGB") for c in catalog]
print(len(catalog), "abaya")

5 abaya



normalize_embeddings, (cosine similarity)

In [ ]:
from sentence_transformers import SentenceTransformer

clip = SentenceTransformer("clip-ViT-B-32")
emb = clip.encode(imgs, normalize_embeddings=True, show_progress_bar=True)
np.save(f"{PROJECT}/index.npy", emb)
print("shape", emb.shape)

modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.03k [00:00<?, ?B/s]

0_CLIPModel/model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

0_CLIPModel/model.safetensors: downloading bytes:           |  0.00B            

... (4 outputs left out)

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

shape (5, 512)


search function

In [ ]:
def find_similar(img, k=3):
    q = clip.encode(img.convert("RGB"), normalize_embeddings=True)
    sims = emb @ q
    return [(catalog[i]["code"], catalog[i]["name"], round(float(sims[i]), 3)) for i in np.argsort(-sims)[:k]]

find_similar(imgs[0])

[('AYL-095', 'عباية بقصة كلوش بقماش نقش الغزال قلبتين', 1.0),
 ('AYL-094', 'عباية بقصة ستريت مع فتحات في الجوانب و زيج في الاطراف', 0.904),
 ('AYL-089', 'عباية بقصة البليزر الخفيف مع اسفنج في الاكتف', 0.848)]

In [ ]:
import random
from PIL import ImageEnhance

def augment(img):
    w, h = img.size
    img = img.crop((int(w * .1), int(h * .1), int(w * .9), int(h * .9)))
    return ImageEnhance.Brightness(img).enhance(random.uniform(.7, 1.3))

top1 = top3 = 0
for i, img in enumerate(imgs):
    q = clip.encode(augment(img), normalize_embeddings=True)
    order = np.argsort(-(emb @ q))
    top1 += int(order[0] == i)
    top3 += int(i in order[:3])

n = len(imgs)
print(f"Top-1: {top1}/{n} = {top1/n:.0%}")
print(f"Top-3: {top3}/{n} = {top3/n:.0%}")

Top-1: 5/5 = 100%
Top-3: 5/5 = 100%
